# Fine-Tuning de modelos con HuggingFace

En este notebook profundizaremos en el **fine-tuning** de modelos preentrenados usando el ecosistema
de HuggingFace. Cubriremos fine-tuning para clasificacion de texto, NER (Named Entity Recognition),
y tecnicas eficientes como **LoRA** (Low-Rank Adaptation).

**Objetivos:**
- Saber cuando conviene fine-tunear vs usar prompting
- Fine-tunear modelos para clasificacion y NER
- Aplicar PEFT/LoRA para fine-tuning eficiente
- Publicar modelos en HuggingFace Hub

## Cuando fine-tunear vs usar prompting

### Fine-tuning
Mejor cuando:
- Tienes **datos etiquetados** (>500 ejemplos idealmente)
- Necesitas **maxima precision** en una tarea especifica
- El **costo de inferencia** debe ser bajo (modelos pequenos fine-tuneados son mas baratos)
- Necesitas **latencia baja** (modelos pequenos son mas rapidos)
- El dominio es **muy especifico** (jerga medica, legal, etc.)

### Prompting (zero-shot / few-shot)
Mejor cuando:
- **No tienes datos** etiquetados
- La tarea requiere **razonamiento complejo**
- Necesitas **flexibilidad** para cambiar la tarea rapidamente
- Es un **prototipo** o prueba de concepto
- La tarea cambia frecuentemente

### Matriz de decision

| Criterio | Prompting | Fine-tuning |
|----------|-----------|-------------|
| Datos etiquetados | < 100 ejemplos | > 500 ejemplos |
| Costo por inferencia | Alto (modelo grande) | Bajo (modelo pequeno) |
| Tiempo de setup | Minutos | Horas-dias |
| Precision | Buena-media | Alta |
| Flexibilidad | Alta | Baja (tarea fija) |
| Mantenimiento | Bajo | Medio (reentrenar) |

## Fine-tuning para clasificacion

Vamos a fine-tunear **DistilBERT** para clasificacion de emociones en texto.
El dataset `emotion` contiene textos etiquetados con 6 emociones:
sadness, joy, love, anger, fear, surprise.

In [ ]:
# !pip install transformers datasets evaluate accelerate

from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np

# Load emotion dataset
dataset = load_dataset("emotion")

# Take a subset for faster training
train_data = dataset["train"].shuffle(seed=42).select(range(2000))
val_data = dataset["validation"].shuffle(seed=42).select(range(500))
test_data = dataset["test"].shuffle(seed=42).select(range(500))

# Emotion labels
label_names = ["sadness", "joy", "love", "anger", "fear", "surprise"]

print(f"Train: {len(train_data)} samples")
print(f"Val:   {len(val_data)} samples")
print(f"Test:  {len(test_data)} samples")
print(f"\nLabels: {label_names}")

# Class distribution
fig, ax = plt.subplots(figsize=(8, 4))
label_counts = {}
for label in train_data["label"]:
    name = label_names[label]
    label_counts[name] = label_counts.get(name, 0) + 1

colors = ["#4c6ef5", "#51cf66", "#ff6b6b", "#ffd43b", "#845ef7", "#ff922b"]
ax.bar(label_counts.keys(), label_counts.values(), color=colors)
ax.set_title("Distribucion de emociones en el dataset de entrenamiento")
ax.set_ylabel("Cantidad")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Show examples
print("\nEjemplos por emocion:")
for label_id, name in enumerate(label_names):
    examples = [t for t, l in zip(train_data["text"], train_data["label"]) if l == label_id]
    if examples:
        print(f"  {name.upper()}: '{examples[0][:80]}...'")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load DistilBERT tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize dataset
def tokenize_function(examples):
    """Tokenize text with truncation and padding."""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length",
    )

# Apply tokenization
tokenized_train = train_data.map(tokenize_function, batched=True)
tokenized_val = val_data.map(tokenize_function, batched=True)
tokenized_test = test_data.map(tokenize_function, batched=True)

# Remove text column and rename label
for ds in [tokenized_train, tokenized_val, tokenized_test]:
    ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load model for 6-class classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),
    id2label={i: name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)},
)

# Show model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Modelo: {model_name}")
print(f"Parametros totales: {total_params:,}")
print(f"Parametros entrenables: {trainable_params:,}")
print(f"Clases: {len(label_names)}")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    """Compute accuracy and weighted F1 score."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_weighted": f1_score(labels, predictions, average="weighted"),
        "f1_macro": f1_score(labels, predictions, average="macro"),
    }

print("Funcion compute_metrics definida.")
print("Metricas: accuracy, f1_weighted, f1_macro")

In [ ]:
from transformers import TrainingArguments, Trainer

# Training arguments
training_args = TrainingArguments(
    output_dir="./results_emotion",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    logging_steps=25,
    report_to="none",
    fp16=False,  # Set True if GPU supports it
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# Train
print("Iniciando entrenamiento...")
train_result = trainer.train()

# Print training summary
print(f"\nEntrenamiento completado!")
print(f"  Tiempo total: {train_result.metrics['train_runtime']:.1f}s")
print(f"  Loss final: {train_result.metrics['train_loss']:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

# Evaluate on test set
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Classification report
print("Reporte de clasificacion:")
print("=" * 60)
print(classification_report(labels, preds, target_names=label_names))

# Confusion matrix
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(len(label_names)))
ax.set_yticks(range(len(label_names)))
ax.set_xticklabels(label_names, rotation=45, ha="right")
ax.set_yticklabels(label_names)
ax.set_xlabel("Prediccion")
ax.set_ylabel("Real")
ax.set_title("Matriz de confusion - Clasificacion de emociones")

# Add values
for i in range(len(label_names)):
    for j in range(len(label_names)):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color=color, fontsize=11)

plt.colorbar(im)
plt.tight_layout()
plt.show()

# Per-class metrics visualization
report = classification_report(labels, preds, target_names=label_names, output_dict=True)
per_class = {name: report[name] for name in label_names}

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(label_names))
width = 0.25

ax.bar(x - width, [per_class[n]["precision"] for n in label_names], width, label="Precision", color="#4c6ef5")
ax.bar(x, [per_class[n]["recall"] for n in label_names], width, label="Recall", color="#51cf66")
ax.bar(x + width, [per_class[n]["f1-score"] for n in label_names], width, label="F1", color="#ff6b6b")

ax.set_xticks(x)
ax.set_xticklabels(label_names, rotation=45)
ax.set_ylabel("Score")
ax.set_title("Metricas por clase")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## Fine-tuning para NER

El **NER** (Named Entity Recognition) es una tarea de **clasificacion a nivel de token**,
donde cada token del texto se clasifica en una categoria de entidad:

- **PER**: Persona
- **ORG**: Organizacion
- **LOC**: Locacion
- **MISC**: Miscelanea
- **O**: No es entidad

Se usa el formato **BIO** (Beginning, Inside, Outside):
- `B-PER`: Inicio de entidad persona
- `I-PER`: Continuacion de entidad persona
- `O`: No es entidad

Un desafio importante es alinear las etiquetas con la tokenizacion subword.

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification
)
import numpy as np

# Load CoNLL-2003 NER dataset
ner_dataset = load_dataset("conll2003", trust_remote_code=True)

# Take a subset for faster training
ner_train = ner_dataset["train"].shuffle(seed=42).select(range(2000))
ner_val = ner_dataset["validation"].shuffle(seed=42).select(range(500))

# NER label names
ner_labels = ner_dataset["train"].features["ner_tags"].feature.names
print(f"NER Labels: {ner_labels}")
print(f"Num labels: {len(ner_labels)}")

# Show an example
example = ner_train[0]
print(f"\nEjemplo:")
print(f"  Tokens: {example['tokens']}")
print(f"  Tags:   {[ner_labels[t] for t in example['ner_tags']]}")

# Load tokenizer
ner_model_name = "distilbert-base-uncased"
ner_tokenizer = AutoTokenizer.from_pretrained(ner_model_name)

def tokenize_and_align_labels(examples):
    """Tokenize and align NER labels with subword tokens.
    
    When a word is split into multiple subword tokens, only the first
    subword gets the original label; the rest get -100 (ignored in loss).
    """
    tokenized = ner_tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
    )
    
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None
        
        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens ([CLS], [SEP], [PAD]) get -100
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # First subword of a word gets the original label
                label_ids.append(label[word_idx])
            else:
                # Subsequent subwords of the same word get -100
                label_ids.append(-100)
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized["labels"] = labels
    return tokenized

# Apply tokenization with label alignment
tokenized_ner_train = ner_train.map(tokenize_and_align_labels, batched=True)
tokenized_ner_val = ner_val.map(tokenize_and_align_labels, batched=True)

print(f"\nTokenization with label alignment complete.")
print(f"  Train: {len(tokenized_ner_train)} examples")
print(f"  Val:   {len(tokenized_ner_val)} examples")

In [ ]:
# !pip install seqeval
import evaluate
from transformers import DataCollatorForTokenClassification

# Load seqeval metric for entity-level evaluation
seqeval_metric = evaluate.load("seqeval")

def compute_ner_metrics(eval_pred):
    """Compute entity-level precision, recall, and F1."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    # Convert IDs back to label names, ignoring -100
    true_labels = []
    true_preds = []
    
    for pred_seq, label_seq in zip(predictions, labels):
        true_label = []
        true_pred = []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:  # Ignore special tokens
                true_label.append(ner_labels[l])
                true_pred.append(ner_labels[p])
        true_labels.append(true_label)
        true_preds.append(true_pred)
    
    results = seqeval_metric.compute(
        predictions=true_preds,
        references=true_labels
    )
    
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Load model for token classification
ner_model = AutoModelForTokenClassification.from_pretrained(
    ner_model_name,
    num_labels=len(ner_labels),
    id2label={i: label for i, label in enumerate(ner_labels)},
    label2id={label: i for i, label in enumerate(ner_labels)},
)

# Data collator for token classification (handles padding of labels)
data_collator = DataCollatorForTokenClassification(tokenizer=ner_tokenizer)

# Training arguments
ner_training_args = TrainingArguments(
    output_dir="./results_ner",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=25,
    report_to="none",
)

# Create Trainer
ner_trainer = Trainer(
    model=ner_model,
    args=ner_training_args,
    train_dataset=tokenized_ner_train,
    eval_dataset=tokenized_ner_val,
    data_collator=data_collator,
    compute_metrics=compute_ner_metrics,
)

# Train
print("Entrenando modelo NER...")
ner_train_result = ner_trainer.train()

# Evaluate
ner_eval = ner_trainer.evaluate()
print(f"\nResultados NER:")
print(f"  Precision: {ner_eval['eval_precision']:.4f}")
print(f"  Recall:    {ner_eval['eval_recall']:.4f}")
print(f"  F1:        {ner_eval['eval_f1']:.4f}")
print(f"  Accuracy:  {ner_eval['eval_accuracy']:.4f}")

## PEFT: Fine-tuning eficiente con LoRA

**LoRA** (Low-Rank Adaptation) es una tecnica de **PEFT** (Parameter-Efficient Fine-Tuning) que
permite fine-tunear modelos grandes modificando solo una fraccion minima de los parametros.

### Como funciona LoRA:
1. **Congela** todos los parametros del modelo original
2. Inserta **matrices de bajo rango** (A y B) junto a las capas de atencion
3. Solo entrena estas matrices nuevas (tipicamente <1% de parametros)

```
W_original (congelado) + delta_W = W_original + B * A
donde B es (d x r) y A es (r x d), con r << d
```

### Ventajas de LoRA:
- **Menos memoria**: Solo entrena ~0.1-1% de parametros
- **Mas rapido**: Menos gradientes que calcular
- **Multitarea**: Puedes tener multiples adaptadores LoRA para un mismo modelo base
- **Sin degradacion**: Rendimiento comparable al fine-tuning completo

In [ ]:
# !pip install peft

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

# Load a fresh base model
base_model_name = "distilbert-base-uncased"
base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=6,  # 6 emotions
)

# Count original parameters
total_params = sum(p.numel() for p in base_model.parameters())
print(f"Modelo base: {base_model_name}")
print(f"Parametros totales: {total_params:,}")

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,       # Sequence classification
    r=8,                               # Rank of the low-rank matrices
    lora_alpha=16,                     # Scaling factor (alpha/r)
    lora_dropout=0.1,                  # Dropout for LoRA layers
    target_modules=["q_lin", "v_lin"], # Which layers to apply LoRA to
    bias="none",                       # Don't train bias terms
)

# Apply LoRA to the model
lora_model = get_peft_model(base_model, lora_config)

# Compare parameters
trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in lora_model.parameters())
reduction = (1 - trainable_params / all_params) * 100

print(f"\nModelo con LoRA:")
print(f"  Parametros totales:      {all_params:,}")
print(f"  Parametros entrenables:  {trainable_params:,}")
print(f"  Reduccion:               {reduction:.1f}%")
print(f"  Ratio:                   {trainable_params/all_params:.4%}")

# Show model architecture with LoRA modules
print(f"\nModulos LoRA:")
lora_model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer
import time

# Use the same tokenized emotion dataset from before
# Training arguments for LoRA (can use larger batch size due to less memory)
lora_training_args = TrainingArguments(
    output_dir="./results_emotion_lora",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=1e-3,  # LoRA typically uses higher learning rate
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    logging_steps=25,
    report_to="none",
)

# Create Trainer with LoRA model
lora_trainer = Trainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# Train with LoRA
print("Entrenando con LoRA...")
start_time = time.time()
lora_result = lora_trainer.train()
lora_time = time.time() - start_time

# Evaluate LoRA model
lora_eval = lora_trainer.evaluate(tokenized_test)

# Compare with full fine-tuning
print(f"\n{'=' * 60}")
print("COMPARACION: Full Fine-tuning vs LoRA")
print(f"{'=' * 60}")
print(f"{'Aspecto':<25} {'Full FT':>15} {'LoRA':>15}")
print(f"{'-' * 55}")
print(f"{'Params entrenables':<25} {trainable_params:>15,} {sum(p.numel() for p in lora_model.parameters() if p.requires_grad):>15,}")
print(f"{'Tiempo entrenamiento':<25} {'N/A':>15} {lora_time:>14.1f}s")
print(f"{'F1 (weighted)':<25} {'(ver arriba)':>15} {lora_eval.get('eval_f1_weighted', 0):>15.4f}")
print(f"{'Accuracy':<25} {'(ver arriba)':>15} {lora_eval.get('eval_accuracy', 0):>15.4f}")

## Subir modelo al Hub

HuggingFace Hub es una plataforma para compartir modelos, datasets y demos.
Subir tu modelo permite:
- **Compartir** con la comunidad o tu equipo
- **Versionado** automatico de modelos
- **Inferencia** gratuita a traves de la API
- **Model cards** para documentar el modelo

In [ ]:
# Push model to HuggingFace Hub
# NOTE: You need to be logged in first with: huggingface-cli login

# --- UNCOMMENT THE LINES BELOW TO ACTUALLY PUSH ---

# Option 1: Push with Trainer
# trainer.push_to_hub(
#     repo_id="your-username/emotion-classifier-distilbert",
#     commit_message="Upload emotion classifier fine-tuned on emotion dataset",
# )

# Option 2: Push model and tokenizer separately
# model.push_to_hub("your-username/emotion-classifier-distilbert")
# tokenizer.push_to_hub("your-username/emotion-classifier-distilbert")

# Option 3: Save locally first, then push
# model.save_pretrained("./my_emotion_model")
# tokenizer.save_pretrained("./my_emotion_model")
# # Then push the saved directory:
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path="./my_emotion_model",
#     repo_id="your-username/emotion-classifier-distilbert",
#     repo_type="model",
# )

# For LoRA models, you can push just the adapter (much smaller):
# lora_model.push_to_hub("your-username/emotion-classifier-lora")
# This only uploads the LoRA weights (~few MB), not the full model.
# To use it later:
# from peft import PeftModel
# base = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=6)
# model = PeftModel.from_pretrained(base, "your-username/emotion-classifier-lora")

print("Ejemplo de como subir modelo al Hub (codigo comentado arriba).")
print("\nPasos:")
print("  1. Instalar: pip install huggingface_hub")
print("  2. Login: huggingface-cli login")
print("  3. Descomentar el codigo de push arriba")
print("\nPara LoRA, solo se suben los adaptadores (~MB vs ~GB del modelo completo).")

## Resumen: arbol de decision de fine-tuning

```
Tengo una tarea de NLP?
|
+-- Es una tarea estandar (sentimiento, NER, resumen)?
|   |
|   +-- SI -> Prueba un pipeline de HuggingFace primero
|   |         Es suficiente? -> SI -> Usa el pipeline
|   |                       -> NO -> Fine-tunea
|   |
|   +-- NO -> Es tarea de generacion/razonamiento?
|             |
|             +-- SI -> Usa un LLM con prompting
|             +-- NO -> Fine-tunea un modelo especifico
|
+-- Tengo datos etiquetados?
    |
    +-- > 10,000 -> Fine-tuning completo
    +-- 500-10,000 -> Fine-tuning con regularizacion fuerte o LoRA
    +-- < 500 -> Few-shot con LLM o data augmentation + fine-tuning
    +-- 0 -> Zero-shot con LLM
```

### Tecnicas de fine-tuning por escenario

| Escenario | Tecnica recomendada | Modelo sugerido |
|-----------|--------------------|-----------------|
| Clasificacion con muchos datos | Full fine-tuning | BERT, DistilBERT |
| Clasificacion con pocos datos | LoRA + augmentation | DistilBERT |
| NER | Fine-tuning completo | BERT, RoBERTa |
| Generacion de texto | LoRA/QLoRA | LLaMA, Mistral |
| Seguir instrucciones | RLHF o DPO | LLaMA, Mistral |
| Multilingue | Fine-tuning | XLM-RoBERTa, mBERT |
| Presupuesto limitado | LoRA + quantizacion | Modelos 7B quantizados |